# Implementation
This section details how MEAMT (Multi-objective evolutionary algorithms with many tables) was implemented utilizing the DEAP library. All the main code is encapsulated on module `meant_core.py`

In [11]:
import inspect
from IPython.display import Code
import meamt_core 

## Enviroment configuration (ToolBox)
In `DEAP`, `toolbox` is the object that allows for creation of custom classes and functions along with pre-existing to facilitate implementation, in this case while `parent selection strategy` is custom-designed, the mutation and crossover operators utilize literature gold standard methods provided by the library. This way `MEAMT` perfomance can be isolated and determined with high certainty.

In [12]:
Code(data=inspect.getsource(meamt_core.build_toolbox), language='python')

def build_toolbox(funcao_avaliacao, ind_size, n_pop, n_obj):
    """Constrói o toolbox do DEAP de forma dinâmica para qualquer benchmark."""
    setup_deap_classes(n_obj)
    
    toolbox = base.Toolbox()
    toolbox.register("attr_float", random.random)
    toolbox.register("individual", tools.initRepeat, creator.Individual, toolbox.attr_float, n=ind_size)
    toolbox.register("population", tools.initRepeat, list, toolbox.individual, n=n_pop)
    
    # Avaliação dinâmica (passada por parâmetro)
    toolbox.register("evaluate", funcao_avaliacao) 
    
    toolbox.register("mate", tools.cxSimulatedBinaryBounded, eta=20.0, low=0.0, up=1.0)
    toolbox.register("mutate", tools.mutPolynomialBounded, eta=20.0, low=0.0, up=1.0, indpb=1.0/ind_size)
    
    return toolbox

## Selection
`MEAMT` selects the parents to reproduction by a tournment of 2, the tournment occours between tables with respect to their `score`, a selected table
choses and individual of its subpopulation randomly and if such an individual generates succeful offspring the `score` of the table is increased by one.

In [13]:
Code(data=inspect.getsource(meamt_core.select_parents), language='python')

def select_parents(tables, num_tables):
    """Selects parents based on table scores."""
    selected = []
    for _ in range(2):
        random1 = random.randint(0, num_tables - 1)
        random2 = random.randint(0, num_tables - 1)

        if len(tables[random1]) == 0: winner = random2
        elif len(tables[random2]) == 0: winner = random1
        elif tables[random1].score >= tables[random2].score:
            winner = random1
        else:
            winner = random2

        ind = random.choice(tables[winner])
        ind.Parent_Table = winner
        selected.append(ind)
    return selected

## Insertion
To insert a new individual in a table we first verify if the new individual combained fitness is better than the worst of said table. This is done in the `Non Denominated` subpop by appending the new individual and selecting all individuals again using `NSGA-II`, if the individual survives, then the score of its parent is increased.
In other tables linear search is used to find the worst individual present in the subpopulation, then its verified if its wheighted fitness is worse than the fitness from the new one, if it is, the worst individual is replaced in the subpop and, as previous, table score is increased.

In [14]:
Code(data=inspect.getsource(meamt_core.insert_in_tables), language='python')

def insert_in_tables(tables, num_tables, off, max_table_size, n_obj):
    """Inserts offspring into the appropriate tables and updates scores."""
    # 1. Insert on table ND (0)
    tabela_nd = tables[0]
    tabela_nd.append(off)
    
    # NSGA2 select
    nova_selecao = tools.selNSGA2(tabela_nd, max_table_size)
    tabela_nd[:] = nova_selecao

    # Se off is in the table -> increase score
    if any(ind is off for ind in tabela_nd):
        if off.Parent_Table is not None:
            tables[off.Parent_Table].score += 1

    # Try to insert in other tables
    for i in range(1, num_tables):
        # Repassando o n_obj aqui
        fit_off = calc_combined_fitness(off, i, n_obj)
        worse_val = -1.0
        worse_idx = -1

        for idx, ind in enumerate(tables[i]):
            # Repassando o n_obj aqui
            fit_ind = calc_combined_fitness(ind, i, n_obj)
            if fit_ind > worse_val:
                worse_val = fit_ind
                worse_idx = idx

        # Minimization: If new is less than the worst, replaces it
        if fit_off < worse_val:
            tables[i][worse_idx] = off
            if off.Parent_Table is not None:
                tables[off.Parent_Table].score += 1

## Main Loop (MEAMT)
The run function brings all these pieces together. It evaluates the generations, distributes the offspring into the correct tables, updates the scores, and, for each generation, calculates the Hypervolume and IGD+ for the logbook.

In [15]:
Code(data=inspect.getsource(meamt_core.run), language='python')

def run(tables, pareto_front_true, num_tables, max_table_size, ngen, toolbox, cxpb, mutpb, ref_point_hv, n_obj):
    """Executa as gerações do algoritmo MEAMT (Versão Paralela)."""
    logbook = tools.Logbook()
    logbook.header = "gen", "hypervolume", "igd_plus"
    
    for gen in range(1, ngen + 1):
        # Reset score
        for t in tables.values():     
            t.score = 0.0

        # =========================================================
        # FASE 1: GERAR TODOS OS FILHOS DA GERAÇÃO
        # =========================================================
        offspring = []
        for _ in range((max_table_size * num_tables) // 2):
            parents = select_parents(tables, num_tables)

            off1, off2 = toolbox.clone(parents[0]), toolbox.clone(parents[1])
            off1.Parent_Table = parents[0].Parent_Table
            off2.Parent_Table = parents[1].Parent_Table

            if random.random() < cxpb:
                toolbox.mate(off1, off2)
                del off1.fitness.values, off2.fitness.values

            if random.random() < mutpb:
                toolbox.mutate(off1)
                del off1.fitness.values
            if random.random() < mutpb:
                toolbox.mutate(off2)
                del off2.fitness.values
                
            offspring.extend([off1, off2])

        # =========================================================
        # FASE 2: AVALIAÇÃO EM PARALELO (A Mágica acontece aqui!)
        # =========================================================
        # Separa apenas os indivíduos que sofreram mutação/crossover e precisam de avaliação
        invalid_ind = [ind for ind in offspring if not ind.fitness.valid]
        
        # O toolbox.map vai distribuir a lista de indivíduos entre os núcleos do seu PC
        fitnesses = toolbox.map(toolbox.evaluate, invalid_ind)
        
        # Atribui os resultados de volta aos indivíduos
        for ind, fit in zip(invalid_ind, fitnesses):
            ind.fitness.values = fit

        # =========================================================
        # FASE 3: INSERÇÃO
        # =========================================================
        for off in offspring:
            insert_in_tables(tables, num_tables, off, max_table_size,n_obj)
            
        # =========================================================
        # MÉTRICAS
        # =========================================================
        hv_val = hypervolume(tables[0], ref_point_hv)
        approx_front = np.array([ind.fitness.values for ind in tables[0]])
        igd_plus_val = calculate_igd_plus(pareto_front_true, approx_front)
        
        logbook.record(gen=gen, hypervolume=hv_val, igd_plus=igd_plus_val)
        
    return logbook